# Exercise 1.3.1.7 — implement the batch-aware intervention hook

> Part of [Delta Drills](https://delta-drills.vercel.app) ARENA practice. When the test cell passes, your completion is reported back to your account automatically.

**Section:** `1.3.1 Linear Probes`  
**Notebook:** `1.3.1_Linear_Probes_exercises.ipynb`  
**Return to Delta Drills:** [https://delta-drills.vercel.app/?arena_exercise=1.3.1.7](https://delta-drills.vercel.app/?arena_exercise=1.3.1.7)


# [1.3.1] Linear Probes (exercises)

> **ARENA [Streamlit Page](https://arena-chapter1-transformer-interp.streamlit.app/11_[1.3.1]_Linear_Probes)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part31_linear_probes/1.3.1_Linear_Probes_exercises.ipynb?t=20260329) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part31_linear_probes/1.3.1_Linear_Probes_solutions.ipynb?t=20260329)**

Please send any problems / bugs on the `#errata` channel in the [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA), and ask any questions on the dedicated channels for this chapter of material.

You can collapse each section so only the headers are visible, by clicking the arrow symbol on the left hand side of the markdown header cells.

Links to all other chapters: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/refs/heads/main/img/header-31.png" width="350">

# Introduction

This exercise set is built around **linear probing**, one of the most important tools in mechanistic interpretability for understanding what information language models represent internally.

We'll look at three papers:

- The [Geometry of Truth](https://arxiv.org/abs/2310.06824) paper by Marks & Tegmark, which shows that LLMs develop linear representations of truth that generalize across diverse datasets and are causally implicated in model outputs.
- The [deception probes paper](https://arxiv.org/abs/2502.03407) from Apollo Research, which extends this from factual truth to *strategic deception detection* - showing probes trained on simple contrastive data can generalize to realistic deception scenarios.
- The [high-stakes interactions paper](https://arxiv.org/abs/2506.10805) (NeurIPS 2025), which trains attention probes to detect whether a user's *request* is high-stakes - a different target from model intent - and shows they match full LLM classifiers at a fraction of the compute cost.

### What is probing?

The core idea: extract internal activations from a model, then train a simple classifier on them. If a *linear* probe can accurately classify some property from the activations, that property is **linearly represented** in the model's internal state.

From the Geometry of Truth paper:

> *"We identify a linear representation of truth that generalizes across several structurally and topically diverse datasets... these representations are not merely associated with truth, but are also causally implicated in the model's output."*

The "causally implicated" part matters a lot: it's not just that we can read off truth from model internals, but that the model actually *uses* these representations when computing outputs. We'll verify this in Section 3.

### Why this matters for safety

If we can reliably detect truth, deception, or intent from model internals, there are direct implications for model monitoring. [Neel Nanda argues](https://www.lesswrong.com/posts/G9HdpyREaCbFJjKu5/it-is-reasonable-to-research-how-to-use-model-internals-in) probes could even be used *during training*:

> *"There are certain things that may be much easier to specify using the internals of the model. For example: Did it do something for the right reasons? Did it only act this way because it knew it was being trained or watched?"*

But this is genuinely controversial. The worry, as [Bronson Schoen puts it](https://www.lesswrong.com/posts/G9HdpyREaCbFJjKu5/it-is-reasonable-to-research-how-to-use-model-internals-in?commentId=CtZnXwZuBgcWsagwn), is that training against probe signals might just teach the model to hide whatever the probe was measuring:

> *"If you train directly against non-obfuscated internals and no longer see bad behavior, the obvious possibility is that now you've just got obfuscated internals."*

For deception probes specifically, the generalization question is especially pointed: we may need to detect sophisticated deception in scenarios we've never seen. As the Apollo paper puts it: *"our monitors will need to exhibit generalization - correctly identifying deceptive text in new types of scenarios."*

### What you'll build

These exercises cover the full pipeline: extracting activations, visualizing with PCA, training probes, validating them causally, and applying them to deception detection. Layer choice, token position, and probe type all matter - part of the point is developing intuition for *why*.

### Models we'll use

Sections 1-3 use `meta-llama/Llama-2-13b-hf` (base model, ~26GB in bfloat16). The Geometry of Truth paper has specific configurations for this model (`probe_layer=14`, `intervene_layer=8`), so our results should closely match theirs. Section 4 switches to `meta-llama/Meta-Llama-3.1-8B-Instruct` (instruct-tuned, ~16GB), needed for the deception detection instructed-pairs methodology.

Both fit comfortably on a single A100. If you have a multi-GPU setup, the 70B variants are worth trying as a bonus; the paper's strongest results are at that scale.

## Content & Learning Objectives

### 1️⃣ Setup & visualizing truth representations

> ##### Learning Objectives
>
> * Extract hidden state activations from specified layers and token positions
> * Implement PCA to visualize high-dimensional activations
> * Observe that truth is linearly separable in activation space - even without supervision
> * Understand which layers best represent truth via a layer sweep

### 2️⃣ Training & comparing probes

> ##### Learning Objectives
>
> * Implement difference-of-means (MM) and logistic regression (LR) probes
> * Compare probe types: accuracy, direction similarity, and what each captures
> * Understand CCS (Contrastive Consistent Search) as an unsupervised alternative and its limitations

### 3️⃣ Causal interventions

> ##### Learning Objectives
>
> * Understand why classification accuracy alone is insufficient - causal evidence is needed
> * Implement activation patching with probe directions to flip model predictions
> * Compare the causal effects of MM vs. LR probe directions
> * Appreciate that MM probes find more causally implicated directions despite lower classification accuracy

### 4️⃣ Probing for Deception

> ##### Learning Objectives
>
> * Construct instructed-pairs datasets following the deception-detection paper's methodology
> * Train deception probes on instruct-tuned models
> * Evaluate whether deception probes generalize to factual truth/falsehood datasets
> * Understand methodological choices that affect replicability

### 5️⃣ Attention Probes for High-Stakes Detection

> ##### Learning Objectives
>
> * Understand what "high-stakes interactions" means as a probe target, and why it differs from probing model intent
> * Extract full-sequence activations (shape `(n, seq, d_model)`) rather than last-token only
> * Implement an attention probe as a `nn.Module` - a single learned query that computes a weighted sum over token positions before classification
> * Compare attention pooling against last-token and mean-pool baselines using AUROC
> * Inspect learned attention weights to understand which parts of a prompt are most diagnostic

## Reading Material

The core papers we'll be replicating are "The Geometry of Truth" and "Detecting Strategic Deception Using Linear Probes" - you should at least skim both before starting, so you understand the basics. The other references give you more context on the probing literature and the open questions around it.

- [The Geometry of Truth: Emergent Linear Structure in Large Language Model Representations of True/False Datasets](https://arxiv.org/abs/2310.06824) by Marks & Tegmark (COLM 2024). Shows that LLMs develop linear representations of truth that generalise across diverse datasets and are causally implicated in model outputs. Read at least the abstract, and sections 1, 2 & 4 (intro, datasets & visualization). You can skip 3, because we won't be doing much of this kind of patching in these exercises.
- [Detecting Strategic Deception Using Linear Probes](https://arxiv.org/abs/2502.03407) by Goldowsky-Dill et al. (Apollo Research, 2025). Extends truth probing to *strategic deception detection*, showing that probes trained on simple contrastive data can generalise to realistic deception scenarios. Section 4 of this exercise set replicates their methodology. Read the abstract and sections 1 & 3 (introduction and methodology).
- [Detecting High-Stakes Interactions with Activation Probes](https://arxiv.org/abs/2506.10805) by McKenzie et al. (NeurIPS 2025). Trains attention probes to detect whether a user's request is high-stakes, matching full LLM classifiers at much lower cost. Section 5 of this exercise set replicates this. Read the abstract and section 2 (methodology), or alternatively just the [EleutherAI post](https://blog.eleuther.ai/attention-probes/).
- [Discovering Latent Knowledge in Language Models Without Supervision](https://arxiv.org/abs/2212.03827) by Burns et al. (ICLR 2023). Introduces Contrastive Consistent Search (CCS), an unsupervised method for finding truth directions without labelled data. We discuss CCS limitations in section 2 but don't implement it in full. Optional reading.

## Setup code

Before running this, you'll need to clone the Geometry of Truth as well as Deception Detection repos into the `exercises` directory:

```bash
cd chapter1_transformer_interp/exercises

git clone https://github.com/saprmarks/geometry-of-truth.git
git clone https://github.com/ApolloResearch/deception-detection.git
```

`Llama-2-13b-hf` is a gated model, so you'll need a HuggingFace access token (as well as requesting access [here](https://huggingface.co/meta-llama/Llama-2-13b-hf)). When you've got access and made a HuggingFace token, create a `.env` file in your `chapter1_transformer_interp/exercises` directory with:

```
HF_TOKEN=hf_your_token_here
```

then the code below will use this token for authentication.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter1_transformer_interp"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import transformer_lens
except:
    %pip install "openai==1.56.1" einops datasets jaxtyping "sae-lens>=4.0.0,<5.0.0" openai tabulate umap-learn hdbscan eindex-callum git+https://github.com/callummcdougall/CircuitsVis.git#subdirectory=python git+https://github.com/callummcdougall/sae_vis.git@callum/v3 transformer_lens==2.17.0

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}

if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import gc
import json
import os
import pickle
import sys
from dataclasses import dataclass
from pathlib import Path

import circuitsvis as cv
import einops
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch as t
from datasets import load_dataset
from dotenv import load_dotenv
from IPython.display import HTML, display
from jaxtyping import Bool, Float
from plotly.subplots import make_subplots
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from torch import Tensor
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

device = t.device("cuda" if t.cuda.is_available() else "cpu")
dtype = t.bfloat16

# Make sure exercises are in the path
chapter = "chapter1_transformer_interp"
section = "part31_linear_probes"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

import part31_linear_probes.tests as tests
import part31_linear_probes.utils as utils

MAIN = __name__ == "__main__"

In [ ]:
# Set up paths to the cloned repos
# Adjust these if your repos are in a different location
GOT_ROOT = exercises_dir / "geometry-of-truth"  # geometry-of-truth repo
DD_ROOT = exercises_dir / "deception-detection"  # deception-detection repo

assert GOT_ROOT.exists(), f"Please clone geometry-of-truth repo to {GOT_ROOT}"
assert DD_ROOT.exists(), f"Please clone deception-detection repo to {DD_ROOT}"

GOT_DATASETS = GOT_ROOT / "datasets"
DD_DATA = DD_ROOT / "data"

### Loading the model

We start with LLaMA-2-13B, a base (not instruction-tuned) model. The Geometry of Truth paper uses this model with `probe_layer=14` and `intervene_layer=8` - we'll use these exact values.

In [ ]:
load_dotenv(dotenv_path=str(exercises_dir / ".env"))
HF_TOKEN = os.getenv("HF_TOKEN")
assert HF_TOKEN, "Please set HF_TOKEN in your chapter1_transformer_interp/exercises/.env file"

In [ ]:
MODEL_NAME = "meta-llama/Llama-2-13b-hf"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=dtype,
    device_map="auto",
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

NUM_LAYERS = len(model.model.layers)
D_MODEL = model.config.hidden_size
# Layer choices from the geometry-of-truth repo config for llama-2-13b. The paper
# found truth representations are concentrated in early-to-mid layers, and identified
# these specific layers via patching experiments (Section 3, "group (b)").
PROBE_LAYER = 14
INTERVENE_LAYER = 8

print(f"Model: {MODEL_NAME}")
print(f"Layers: {NUM_LAYERS}, Hidden dim: {D_MODEL}")
print(f"Probe layer: {PROBE_LAYER}, Intervene layer: {INTERVENE_LAYER}")

### Loading the datasets

The Geometry of Truth paper uses several carefully curated datasets of simple true/false statements. Each dataset has a `statement` column and a `label` column (1=true, 0=false).

From the paper:
> *"We find that the truth-related structure in LLM representations is much cleaner for our curated datasets than for our unstructured ones."*

Let's load three of these curated datasets and examine them:

In [ ]:
DATASET_NAMES = ["cities", "sp_en_trans", "larger_than"]

datasets = {}
for name in DATASET_NAMES:
    df = pd.read_csv(GOT_DATASETS / f"{name}.csv")
    datasets[name] = df
    print(f"\n{name}: {len(df)} statements ({df['label'].sum()} true, {(1 - df['label']).sum():.0f} false)")
    display(df.head(4))

# 1️⃣ Setup & visualizing truth representations

> ##### Learning Objectives
>
> * Extract hidden state activations from specified layers and token positions
> * Implement PCA to visualize high-dimensional activations
> * Observe that truth is linearly separable in activation space - even without supervision
> * Understand which layers best represent truth via a layer sweep

## Extracting activations

Our first task is to extract hidden state activations from the model. For the Geometry of Truth approach, we extract the **last-token** activation at each specified layer. For declarative statements like "The city of Paris is in France.", the model's representation of whether the statement is true or false is concentrated at the final token position.

Note - the Geometry of Truth paper probes specifically at the **end-of-sentence punctuation** token (the period / full stop). The datasets are designed so that every statement ends with a period, meaning the last token is always the period. This is important because the model's truth representation builds up over the sentence and is concentrated at the final punctuation mark.

A few technical details to keep in mind. We use `output_hidden_states=True` in the forward pass to get all layer activations. `outputs.hidden_states` has length `num_layers + 1`: index 0 is the embedding output, and index `i` for `i >= 1` is the output of layer `i-1`. We also need to handle **padding** correctly, since statements have different lengths. We pad them but must extract the activation at the last *real* (non-padding) token, not the last position.

## Connect to Delta Drills

Paste your Delta Drills auth token below so this exercise can report its completion back to your account.
You can copy the token from your Delta Drills account page.


In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_EXERCISE_ID = "1.3.1.7"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"


### Prior-exercise solutions (auto-imported)

These were imported from ARENA's reference `solutions.py` so you can jump straight into this exercise without having implemented every predecessor. Re-implement them yourself if you'd rather build top-to-bottom.


In [ ]:
from part31_linear_probes.solutions import extract_activations, get_pca_components, layer_sweep_accuracy, MMProbe, LRProbe, compute_generalization_matrix, few_shot_evaluate


### Exercise - implement the batch-aware intervention hook

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵🔵
> 
> You should spend up to 15-20 minutes on this exercise.
> This is the most important exercise in this section - it establishes causality.
> ```

We've provided the scaffolding for `intervention_experiment` below. Your task is to implement `make_batch_hook` - the function that returns a hook which adds the scaled direction vector to hidden states at the right token positions for each batch element.

The hook needs to work with variable-length sequences, since after padding different sequences in a batch end at different positions. Use `attention_mask.sum(dim=1)` to find the real sequence length `end` for each batch element, and `len_suffix` (the number of tokens in " This statement is:") to find the two target positions: `end - len_suffix - 1` (the final period of the statement) and `end - len_suffix` (the first token of " This statement is:", i.e. the word "This").

Look at the simpler `make_intervention_hook` above for reference on handling tuple vs. plain tensor outputs.

Read through the full function before you start. The scaffolding:
* Constructs queries by prepending the few-shot prompt and appending " This statement is:" to each statement
* Registers hooks on each intervention layer (and removes them in a `finally` block so errors don't leave hooks stuck)
* Extracts P(TRUE) - P(FALSE) from the last-token logits after the forward pass

In [ ]:
def intervention_experiment(
    statements: list[str],
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    direction: Float[Tensor, " d_model"],
    few_shot_prompt: str,
    true_id: int,
    false_id: int,
    intervene_layers: list[int],
    intervention: str = "none",
    batch_size: int = 32,
) -> Float[Tensor, " n"]:
    """
    Run the intervention experiment.

    Args:
        statements: Statements to evaluate.
        model: Language model.
        tokenizer: Tokenizer.
        direction: The (already scaled) truth direction vector.
        few_shot_prompt: Few-shot prefix.
        true_id: Token ID for " TRUE".
        false_id: Token ID for " FALSE".
        intervene_layers: List of layer indices to intervene at.
        intervention: "none", "add", or "subtract".
        batch_size: Batch size.

    Returns:
        P(TRUE) - P(FALSE) for each statement.
    """
    assert intervention in ["none", "add", "subtract"]

    # Determine how many tokens " This statement is:" adds
    suffix_tokens = tokenizer.encode(" This statement is:")
    len_suffix = len(suffix_tokens)

    p_diffs = []
    for i in range(0, len(statements), batch_size):
        batch = statements[i : i + batch_size]
        queries = [few_shot_prompt + stmt + " This statement is:" for stmt in batch]

        inputs = tokenizer(queries, return_tensors="pt", padding=True, truncation=True, max_length=512).to(model.device)

        # Register hooks for intervention
        hooks = []
        if intervention != "none":
            dir_device = direction.to(model.device)
            scale = 1.0 if intervention == "add" else -1.0

            # Each sequence in the batch can have a different length, so we iterate over batch
            # elements inside the hook, using attention_mask to find real sequence lengths.
            def make_batch_hook(dir_vec, attn_mask, scl):
                def hook_fn(module, input, output):
                    # YOUR CODE HERE - implement the batch-aware hook:
                    # 1. Extract hidden_states from output (handle tuple or plain tensor)
                    # 2. For each batch element b, find end = attn_mask[b].sum()
                    # 3. Patch at positions end - len_suffix and end - len_suffix - 1
                    # 4. Return the modified output (keeping the tuple structure if applicable)
                    raise NotImplementedError()
                return hook_fn

            for layer_idx in intervene_layers:
                hook = model.model.layers[layer_idx].register_forward_hook(
                    make_batch_hook(dir_device, inputs["attention_mask"], scale)
                )
                hooks.append(hook)

        with t.no_grad():
            # Common pattern for hooks, so failed hooks don't get stuck
            try:
                outputs = model(**inputs)
            finally:
                for hook in hooks:
                    hook.remove()

            # Get logits at the last non-padding position, then get probability differences
            last_idx = inputs["attention_mask"].sum(dim=1) - 1
            batch_indices = t.arange(len(batch), device=outputs.logits.device)
            last_logits = outputs.logits[batch_indices, last_idx]
            probs = last_logits.softmax(dim=-1)
            p_diff = probs[:, true_id] - probs[:, false_id]
            p_diffs.append(p_diff.cpu().float())

    return t.cat(p_diffs)


# Train the intervention probe on cities + neg_cities combined. The paper found that
# "training on statements and their opposites improves generalization" - using both
# a statement and its negation gives the probe a cleaner truth direction.
# Load neg_cities for this paired training
neg_cities_df = pd.read_csv(GOT_DATASETS / "neg_cities.csv")
neg_cities_stmts = neg_cities_df["statement"].tolist()
neg_cities_labels = t.tensor(neg_cities_df["label"].values, dtype=t.float32)

neg_cities_acts_dict = extract_activations(neg_cities_stmts, model, tokenizer, [PROBE_LAYER])
neg_cities_acts = neg_cities_acts_dict[PROBE_LAYER]

# Train probe on cities + neg_cities combined
combined_acts = t.cat([activations["cities"], neg_cities_acts])
combined_labels = t.cat([labels_dict["cities"], neg_cities_labels])
combined_probe = MMProbe.from_data(combined_acts, combined_labels)

# Scale the direction
direction = combined_probe.direction
direction_hat = direction / direction.norm()
true_acts = combined_acts[combined_labels == 1]
false_acts = combined_acts[combined_labels == 0]
true_mean = true_acts.mean(0)
false_mean = false_acts.mean(0)
projection_diff = ((true_mean - false_mean) @ direction_hat).item()
scaled_direction = projection_diff * direction_hat

# Intervene at all layers from INTERVENE_LAYER through PROBE_LAYER. This matches
# the paper's "group (b)" hidden states that were found to be causally implicated.
intervene_layer_list = list(range(INTERVENE_LAYER, PROBE_LAYER + 1))

# Run for all 3 conditions × 2 subsets
results_intervention = {}
for intervention_type in ["none", "add", "subtract"]:
    for subset in ["true", "false"]:
        mask = sp_eval_labels == (1 if subset == "true" else 0)
        subset_stmts = [s for s, m in zip(sp_eval_stmts, mask.tolist()) if m]
        p_diffs = intervention_experiment(
            subset_stmts,
            model,
            tokenizer,
            scaled_direction,
            FEW_SHOT_PROMPT,
            TRUE_ID,
            FALSE_ID,
            intervene_layer_list,
            intervention=intervention_type,
        )
        results_intervention[(intervention_type, subset)] = p_diffs.mean().item()

# Print results
intervention_df = pd.DataFrame(
    {
        "Intervention": ["none", "add", "subtract"],
        "True Stmts (mean P_diff)": [
            f"{results_intervention[('none', 'true')]:.4f}",
            f"{results_intervention[('add', 'true')]:.4f}",
            f"{results_intervention[('subtract', 'true')]:.4f}",
        ],
        "False Stmts (mean P_diff)": [
            f"{results_intervention[('none', 'false')]:.4f}",
            f"{results_intervention[('add', 'false')]:.4f}",
            f"{results_intervention[('subtract', 'false')]:.4f}",
        ],
    }
)
print("\nIntervention results (mean P(TRUE) - P(FALSE)):")
display(intervention_df)

# Grouped bar chart
fig = go.Figure()
for subset, color in [("true", "blue"), ("false", "red")]:
    vals = [results_intervention[(interv, subset)] for interv in ["none", "add", "subtract"]]
    fig.add_trace(
        go.Bar(
            name=f"{subset.capitalize()} statements",
            x=["None", "Add", "Subtract"],
            y=vals,
            marker_color=color,
            opacity=0.7,
        )
    )
fig.update_layout(
    title="Causal Intervention: Effect on P(TRUE) - P(FALSE)",
    yaxis_title="Mean P(TRUE) - P(FALSE)",
    barmode="group",
    height=400,
    width=600,
)
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.show()

<details><summary>Solution</summary>

```python
def intervention_experiment(
    statements: list[str],
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    direction: Float[Tensor, " d_model"],
    few_shot_prompt: str,
    true_id: int,
    false_id: int,
    intervene_layers: list[int],
    intervention: str = "none",
    batch_size: int = 32,
) -> Float[Tensor, " n"]:
    """
    Run the intervention experiment.

    Args:
        statements: Statements to evaluate.
        model: Language model.
        tokenizer: Tokenizer.
        direction: The (already scaled) truth direction vector.
        few_shot_prompt: Few-shot prefix.
        true_id: Token ID for " TRUE".
        false_id: Token ID for " FALSE".
        intervene_layers: List of layer indices to intervene at.
        intervention: "none", "add", or "subtract".
        batch_size: Batch size.

    Returns:
        P(TRUE) - P(FALSE) for each statement.
    """
    assert intervention in ["none", "add", "subtract"]

    # Determine how many tokens " This statement is:" adds
    suffix_tokens = tokenizer.encode(" This statement is:")
    len_suffix = len(suffix_tokens)

    p_diffs = []
    for i in range(0, len(statements), batch_size):
        batch = statements[i : i + batch_size]
        queries = [few_shot_prompt + stmt + " This statement is:" for stmt in batch]

        inputs = tokenizer(queries, return_tensors="pt", padding=True, truncation=True, max_length=512).to(model.device)

        # Register hooks for intervention
        hooks = []
        if intervention != "none":
            dir_device = direction.to(model.device)
            scale = 1.0 if intervention == "add" else -1.0

            # Each sequence in the batch can have a different length, so we iterate over batch
            # elements inside the hook, using attention_mask to find real sequence lengths.
            def make_batch_hook(dir_vec, attn_mask, scl):
                def hook_fn(module, input, output):
                    hidden_states = output[0] if isinstance(output, tuple) else output

                    seq_lens = attn_mask.sum(dim=1)  # [batch]
                    for b in range(hidden_states.shape[0]):
                        end = seq_lens[b].item()
                        for offset in [-len_suffix, -len_suffix - 1]:
                            pos = int(end + offset)
                            if 0 <= pos < hidden_states.shape[1]:
                                hidden_states[b, pos, :] += scl * dir_vec

                    return (hidden_states,) + output[1:] if isinstance(output, tuple) else hidden_states

                return hook_fn

            for layer_idx in intervene_layers:
                hook = model.model.layers[layer_idx].register_forward_hook(
                    make_batch_hook(dir_device, inputs["attention_mask"], scale)
                )
                hooks.append(hook)

        with t.no_grad():
            # Common pattern for hooks, so failed hooks don't get stuck
            try:
                outputs = model(**inputs)
            finally:
                for hook in hooks:
                    hook.remove()

            # Get logits at the last non-padding position, then get probability differences
            last_idx = inputs["attention_mask"].sum(dim=1) - 1
            batch_indices = t.arange(len(batch), device=outputs.logits.device)
            last_logits = outputs.logits[batch_indices, last_idx]
            probs = last_logits.softmax(dim=-1)
            p_diff = probs[:, true_id] - probs[:, false_id]
            p_diffs.append(p_diff.cpu().float())

    return t.cat(p_diffs)
```
</details>

The key result: **adding** the truth direction to false-statement activations should push P(TRUE) - P(FALSE) upward (making the model more likely to predict TRUE), while **subtracting** it from true-statement activations should push it downward. This demonstrates that the probe direction is *causally implicated* in the model's computation, not merely correlated with truth.

### Comparing MM vs. LR interventions

Now let's repeat the intervention experiment using the LR probe's direction instead of the MM direction. We'll scale both directions the same way and compare the Natural Indirect Effects (NIEs).

As a reminder, the NIE for "add" on false statements = P_diff(add) - P_diff(none). A higher NIE means the direction is more causally implicated. Run the code below to see how the two probe types compare.

In [ ]:
# Train LR probe on same data
lr_combined = LRProbe.from_data(combined_acts, combined_labels)
lr_direction = lr_combined.direction.detach()
lr_direction_hat = lr_direction / lr_direction.norm()
lr_proj_diff = ((true_mean - false_mean) @ lr_direction_hat).item()
lr_scaled_direction = lr_proj_diff * lr_direction_hat

# Run intervention for LR direction
lr_results = {}
for intervention_type in ["none", "add", "subtract"]:
    for subset in ["true", "false"]:
        mask = sp_eval_labels == (1 if subset == "true" else 0)
        subset_stmts = [s for s, m in zip(sp_eval_stmts, mask.tolist()) if m]
        p_diffs = intervention_experiment(
            subset_stmts,
            model,
            tokenizer,
            lr_scaled_direction,
            FEW_SHOT_PROMPT,
            TRUE_ID,
            FALSE_ID,
            intervene_layer_list,
            intervention=intervention_type,
        )
        lr_results[(intervention_type, subset)] = p_diffs.mean().item()

# Compute NIEs
mm_nie_false = results_intervention[("add", "false")] - results_intervention[("none", "false")]
mm_nie_true = results_intervention[("subtract", "true")] - results_intervention[("none", "true")]
lr_nie_false = lr_results[("add", "false")] - lr_results[("none", "false")]
lr_nie_true = lr_results[("subtract", "true")] - lr_results[("none", "true")]

nie_df = pd.DataFrame(
    {
        "Probe": ["MM", "MM", "LR", "LR"],
        "Intervention": ["Add to false", "Subtract from true", "Add to false", "Subtract from true"],
        "NIE": [f"{mm_nie_false:.4f}", f"{mm_nie_true:.4f}", f"{lr_nie_false:.4f}", f"{lr_nie_true:.4f}"],
    }
)
print("Natural Indirect Effects (NIE):")
display(nie_df)

# Side-by-side bar chart
fig = go.Figure()
fig.add_trace(
    go.Bar(
        name="MM Probe",
        x=["Add→False", "Sub→True"],
        y=[mm_nie_false, mm_nie_true],
        marker_color="blue",
        opacity=0.7,
    )
)
fig.add_trace(
    go.Bar(
        name="LR Probe",
        x=["Add→False", "Sub→True"],
        y=[lr_nie_false, lr_nie_true],
        marker_color="orange",
        opacity=0.7,
    )
)
fig.update_layout(
    title="Natural Indirect Effect: MM vs LR Probe Directions",
    yaxis_title="NIE (change in P(TRUE)-P(FALSE))",
    barmode="group",
    height=400,
    width=600,
)
fig.show()

<details>
<summary>Question - Which probe type produces a more causally implicated direction? Why might this be?</summary>

The MM (difference-of-means) probe should produce a direction with **higher NIE** than the LR (logistic regression) probe, even though LR achieves higher classification accuracy. From the Geometry of Truth paper:

> *"Mass-mean probe directions are highly causal, with MM outperforming LR and CCS in 7/8 experimental conditions, often substantially."*

The explanation: LR optimizes for *classification accuracy*, which means it can exploit *any* feature that correlates with truth, even if that feature isn't causally used by the model. The MM direction, by contrast, is the *geometric center* of the true/false clusters. As the paper notes: *"In some cases, however, the direction identified by LR can fail to reflect an intuitive best guess for the feature direction, even in the absence of confounding features."*

For a related but slightly different perspective, see the [Adversarial Examples Are Not Bugs, They Are Features](https://arxiv.org/abs/1905.02175) paper, which uses the "feature robustness framing" to explain how the direction learned by an accuracy-optimizing classifier might not always line up with the mean-difference direction, or what we would view as the "canonical" direction of the feature we're trying to learn. (Incidentally, this is also a motivator for why encoders and decoders in SAEs are untied - we use encoder vectors for detection but decoder vectors for steering.)

<img src="https://i.snipboard.io/nO0M5S.jpg" width="400">

This is an important cautionary tale: **high probe accuracy does not guarantee causal relevance**. Always validate with interventions!
</details>

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

def _dd_report_complete():
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    try:
        body = _dd_json.dumps({
            'exercise_id': DD_EXERCISE_ID,
            'passed': True,
        }).encode('utf-8')
        req = _dd_req.Request(
            f'{DD_BACKEND_URL}/api/arena/complete',
            data=body,
            headers={
                'Content-Type': 'application/json',
                'Authorization': f'Bearer {DD_TOKEN}',
            },
            method='POST',
        )
        with _dd_req.urlopen(req, timeout=3) as r:
            r.read()
        print(f'[Delta Drills] reported completion of {DD_EXERCISE_ID}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

# This exercise has no automatic test — call `_dd_report_complete()`
# in a new cell once you're satisfied with your answer.
